# Datarensing: arbeidstimer for studenter 18-29 år (Jørgen)

In [1]:
from pathlib import Path

import pandas as pd

# Fungerer både fra prosjektmappen og fra notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW_PATH = ROOT / "data" / "raw" / "adult24.csv"
OUT_PATH = ROOT / "data" / "processed" / "jorgen_clean.csv"

# NHIS-navn -> beskrivende navn
RENAME = {
    "HHX": "household_id",
    "AGEP_A": "age",
    "WTFA_A": "survey_weight",
    "SCHCURENR_A": "currently_in_school",
    "EMPWRKLSW1_A": "worked_last_week",
    "EMPWKHRS3_A": "usual_weekly_hours",
}

df = pd.read_csv(RAW_PATH, usecols=list(RENAME)).rename(columns=RENAME)
print(df.shape)  # forventet (32629, 6)

(32629, 6)


## Utvalg: studenter 18-29 år
Raden fjernes bare når variabelen er nødvendig for utvalget. Ukjent skolestatus (7/8/9) kan ikke avgjøres, så de utelates.

In [2]:
students = df[df["age"].between(18, 29)]  # fjerner også aldersskodene 97/99
print("Etter alder:", len(students))  # forventet 4086

unknown_school = students["currently_in_school"].isin([7, 8, 9]).sum()
students = students[students["currently_in_school"] == 1].copy()
print("Ukjent skolestatus, utelatt:", unknown_school)  # forventet 147
print("Etter skole:", len(students))  # forventet 1085

Etter alder: 4086
Ukjent skolestatus, utelatt: 147
Etter skole: 1085


## Arbeidstimer per uke
- "Vet ikke"-koder (97/98/99) blir manglende (NaN), og raden beholdes.
- Tomt felt betyr "ikke spurt": de som ikke jobbet forrige uke (kode 2) får 0 timer.
- Ukjent arbeidsstatus (kode 9) får ikke 0 timer, de forblir manglende.

In [3]:
hours = students["usual_weekly_hours"]

dont_know = hours.isin([97, 98, 99])
print("Timer satt til manglende (97/98/99):", dont_know.sum())  # forventet 0
hours = hours.mask(dont_know)

did_not_work = students["worked_last_week"] == 2
print("Ikke jobbet, satt til 0 timer:", did_not_work.sum())  # forventet 418
hours = hours.mask(did_not_work, 0)

students["weekly_work_hours"] = hours.astype("Int64")

Timer satt til manglende (97/98/99): 0
Ikke jobbet, satt til 0 timer: 418


## Kontroll

In [4]:
clean = students[["household_id", "age", "survey_weight", "weekly_work_hours"]]

print("Rader:", len(clean))                                                       # forventet 1085
print("Tomme verdier per kolonne:")
print(clean.isna().sum().to_string())                                             # alle 0
print("Ukjent arbeidsstatus (kode 9):", (students["worked_last_week"] == 9).sum())  # 0
print("Spesialkoder igjen i timer:", clean["weekly_work_hours"].isin([97, 98, 99]).sum())  # 0
print("Alder utenfor 18-29:", (~clean["age"].between(18, 29)).sum())              # 0
print("household_id unik:", clean["household_id"].is_unique)                      # True
print("Maks timer:", clean["weekly_work_hours"].max())                            # 95

Rader: 1085
Tomme verdier per kolonne:
household_id         0
age                  0
survey_weight        0
weekly_work_hours    0
Ukjent arbeidsstatus (kode 9): 0
Spesialkoder igjen i timer: 0
Alder utenfor 18-29: 0
household_id unik: True
Maks timer: 95


## Lagre

In [ ]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(OUT_PATH, index=False)
print("Lagret", clean.shape, "til", OUT_PATH)

Lagret (1085, 4) til c:\School\Data Science\Sleep-Analysis\data\processed\jorgen_clean.csv


: 